# Support Vector Machines & Kernels
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/svm_basics.ipynb)

An SVM finds the hyperplane with the maximum margin between classes. The kernel trick maps data to higher dimensions so non-linear boundaries become linear.

**Covered:** margin intuition, kernel comparison, C and gamma effects, decision boundaries.

## 1. Non-linear toy problem: two moons

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=400, noise=0.22, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler().fit(Xtr)          # SVMs NEED scaled inputs
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

## 2. Kernel comparison

In [ ]:
kernels = {"linear": {}, "poly": {"degree": 3}, "rbf": {}}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2, 2, 300))

for ax, (name, kw) in zip(axes, kernels.items()):
    clf = SVC(kernel=name, C=1.0, **kw).fit(Xtr_s, ytr)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(Xte_s[:, 0], Xte_s[:, 1], c=yte, cmap="coolwarm", s=12)
    ax.set_title(f"{name}  acc={clf.score(Xte_s, yte):.2f}")
plt.suptitle("Kernel trick: same algorithm, different feature space"); plt.show()

## 3. What C and gamma control (RBF)

In [ ]:
Cs = [0.1, 1, 10]; gammas = [0.1, 1, 10]
fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for i, C in enumerate(Cs):
    for j, g in enumerate(gammas):
        clf = SVC(kernel="rbf", C=C, gamma=g).fit(Xtr_s, ytr)
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        axes[i, j].contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
        axes[i, j].scatter(Xtr_s[:, 0], Xtr_s[:, 1], c=ytr, cmap="coolwarm", s=8)
        axes[i, j].set_title(f"C={C}, gamma={g}  acc={clf.score(Xte_s, yte):.2f}", fontsize=9)
plt.suptitle("gamma: boundary wiggliness | C: tolerance of margin errors"); plt.show()

## Key takeaways
- Scale features first - SVM distance geometry assumes comparable units.
- Small `gamma` = smooth boundary (underfit risk); large `gamma` = islands around points (overfit risk).
- Start with `SVC(kernel='rbf')`, tune C/gamma via `GridSearchCV`.
- Probabilities need `probability=True` (slower - internal CV).
- For >50k rows prefer `LinearSVC` or SGDClassifier.